In [ ]:
import pandas as pd
import re
from textblob import TextBlob

# Load CSV
df = pd.read_csv("youtube_videos.csv")

# Clickbait phrases
clickbait_phrases = [
    "you won't believe", "shocking", "unbelievable", "must watch",
    "gone wrong", "top", "secret", "exposed", "this will blow your mind",
    "omg", "what happens next", "surprising"
]

def parse_duration(duration):
    # If ISO 8601 format
    try:
        import isodate
        return isodate.parse_duration(duration).total_seconds()
    except:
        return float(duration)

def label_video(row):
    title = str(row['title']).lower()
    desc = str(row['description']).lower()
    tags = str(row.get('tags', '')).lower()
    
    views = row.get('view_count', 0)
    likes = row.get('like_count', 0)
    comments = row.get('comment_count', 0)
    duration = parse_duration(row.get('duration', 0))

    score = 0

    # Rule 1: Clickbait phrases
    for phrase in clickbait_phrases:
        if phrase in title:
            score += 2

    # Rule 2: Excess punctuation
    if title.count("!") >= 2 or title.count("?") >= 2:
        score += 1

    # Rule 3: ALL CAPS
    if title.isupper():
        score += 1

    # Rule 4: Numbers in title (Top 5, 3 tricks)
    if re.search(r"\d+", title):
        score += 1

    # Rule 5: Very short description
    if len(desc.split()) < 6:
        score += 1

    # Rule 6: High sentiment polarity
    polarity = TextBlob(title).sentiment.polarity
    if abs(polarity) > 0.6:
        score += 1

    # Rule 7: Short video + shocking title
    if duration < 60 and score >= 2:
        score += 1

    # Threshold
    return 1 if score >= 3 else 0


df['clickbait_label'] = df.apply(label_video, axis=1)

# Save labeled CSV
df.to_csv("labeled_videos.csv", index=False)

print("Labeling completed. File saved as labeled_videos.csv")

✅ Labeling completed. File saved as labeled_videos.csv
